# Modern decoder-only transformer, English to German

Train this notebook on its own, after or before the Vaswani notebook. It does not train the other model.

The stack is Pre-RMSNorm, RoPE, grouped-query attention, QK-norm, and a SwiGLU mixture of experts.

Use a **T4 GPU** runtime. Paste the same two tokens in the token cell. Validation BLEU, perplexity, chrF++, and METEOR go to the W&B project `transformer-scratch` under the run `modern_en_de`. The test set is not sent to W&B, and COMET is not installed. The model page [wandererupak/modern-transformer-en-de](https://huggingface.co/wandererupak/modern-transformer-en-de) links those charts. The Drive folder is shared with the Vaswani notebook, so the data cell reuses the corpus if it is already there.

In [ ]:
import torch
print('torch', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU, then rerun.')
print('GPU', torch.cuda.get_device_name(0))

In [ ]:
import os
REPO = 'https://github.com/rupaktiwari/transformer-scratch.git'
if not os.path.exists('/content/transformer-scratch/mt'):
    !git clone {REPO} /content/transformer-scratch
%cd /content/transformer-scratch
!git pull
!pip install -q uv
!uv pip install --system -r requirements-colab.txt

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/transformer-scratch'
DATA = ROOT + '/data'
SAVE = ROOT + '/checkpoints/modern_en_de'
os.makedirs(DATA, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)

In [ ]:
import os
if not os.path.exists(DATA + '/meta.json'):
    !python -m mt.prepare_data --train-samples 200000 --out {DATA}
else:
    print('Reusing the corpus already on Drive')

In [ ]:
# Paste your tokens in the quotes. Do not commit or push this notebook after that.
WANDB_TOKEN = ""
HF_TOKEN = ""

import os
os.environ["WANDB_API_KEY"] = WANDB_TOKEN
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO_ID"] = "wandererupak/modern-transformer-en-de"

import wandb
from huggingface_hub import login
wandb.login(key=WANDB_TOKEN)
login(token=HF_TOKEN)
print("Charts will be linked from https://huggingface.co/wandererupak/modern-transformer-en-de")

In [ ]:
!python -m mt.train --config configs/modern_en_de.yaml --data-dir {DATA} --save-dir {SAVE}

In [ ]:
!python -m mt.evaluate --config configs/modern_en_de.yaml --data-dir {DATA} --checkpoint {SAVE}/best.pt --skip-comet